In [ ]:
!pip install opencv-python-headless Pillow numpy matplotlib

In [ ]:
import cv2
import numpy as np
import io
import os


In [ ]:
from PIL import Image

In [ ]:
def compute_ela(image_path: str, quality: int = 90) -> np.ndarray:

    original = Image.open(image_path).convert("RGB") # first conv to rgb

    # Re-save at specified JPEG quality
    buffer = io.BytesIO()
    original.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    resaved = Image.open(buffer).convert("RGB") # resaved image

    # Compute absolute difference
    original_arr = np.array(original, dtype=np.float32) # original
    resaved_arr  = np.array(resaved,  dtype=np.float32) # re-saved
    ela = np.abs(original_arr - resaved_arr) # original image - rev

    # Normalize to 0-255 for visibility
    ela = ela * (255.0 / max(ela.max(), 1e-6))
    ela = ela.astype(np.uint8)

    return ela


In [ ]:
def ela_suspicion_score(ela_image: np.ndarray) -> float:
    """(0.0 = clean, 1.0 = highly suspicious)"""
    gray = cv2.cvtColor(ela_image, cv2.COLOR_RGB2GRAY)
    mean_brightness = gray.mean()
    score = min(mean_brightness / 50.0, 1.0)
    return round(float(score), 4)


In [ ]:
def ela_prefilter(image_path: str, threshold: float = 0.3) -> dict:
    """Run full ELA pipeline and return results."""
    ela = compute_ela(image_path)
    score = ela_suspicion_score(ela)
    return {
        "ela_image":     ela,
        "score":         score,
        "is_suspicious": score >= threshold, # score <-> threshold
    }


In [ ]:
import matplotlib.pyplot as plt
from google.colab import files

In [ ]:
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print("Uploaded:", image_path)

In [ ]:
result = ela_prefilter(image_path)

print(f"ELA Score:    {result['score']}")
print(f"Suspicious:   {result['is_suspicious']}")

# Show original and ELA side by side
original = np.array(Image.open(image_path).convert("RGB"))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(original)
axes[0].set_title("Original Document", fontsize=14)
axes[0].axis("off")

axes[1].imshow(result["ela_image"])
axes[1].set_title(f"ELA Output  |  score={result['score']}  |  suspicious={result['is_suspicious']}", fontsize=14)
axes[1].axis("off")

plt.tight_layout()
plt.savefig("ela_result.png", dpi=150)
plt.show()
print("Saved ela_result.png")